In [25]:
import pandas as pd
import json

`rope_scaling`'s beta_fast field must be a float, got 32
`rope_scaling`'s beta_slow field must be a float, got 1


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

: 

: 

: 

In [26]:
import pandas as pd
import json
from pathlib import Path
import os

base_dir = Path("finetuning_eval")
json_files = list(base_dir.rglob("metrics.json"))
print(json_files)
all_dfs = []

for json_file in json_files:
    with open(json_file, "r") as f:
        data = json.load(f)

    scores_list = []
    for entry in data.get("all_primary_scores", []):
        alias, score = entry.rsplit(": ", 1)
        scores_list.append({"alias": alias, "primary_score": float(score)})
    df_primary_scores = pd.DataFrame(scores_list)

    tasks_list = []
    for task in data.get("tasks", []):
        entry = {"alias": task["alias"], "num_instances": task.get("num_instances"), "task_suite": json_file.parts[1]}
        entry.update(task.get("metrics", {}))
        tasks_list.append(entry)
    df_tasks = pd.DataFrame(tasks_list)

    df = pd.merge(df_primary_scores, df_tasks, on="alias", how="left")
    df["model"] = os.path.basename(json_file.parent)
    all_dfs.append(df)

df = pd.concat(all_dfs, ignore_index=True)

def rename(name):
    if "arc_easy" in name:
        return "ARC\_E"
    elif "arc_challenge" in name:
        return "ARC\_C"
    elif "boolq" in name:
        return "BoolQ"
    elif "csqa" in name:
        return "CSQA"
    elif "hellaswag" in name:
        return "HSwag"
    elif "mmlu_pro" in name:
        return "MMLU-Pro"
    elif "mmlu" in name:
        return "MMLU"
    elif "openbookqa" in name:
        return "OBQA"
    elif "piqa" in name:
        return "PIQA"
    elif "socialiqa" in name:
        return "SIQA"
    elif "winogrande" in name:
        return "WinoG"
    elif "agi" in name:
        return "AGIEval"
    elif "bbh" in name:
        return "BBH"
    elif "gsm8k" in name:
        return "GSM8K"
    elif "triviaqa" in name:
        return "TriviaQA"
    elif "coqa" in name:
        return "CoQA"
    elif "drop" in name:
        return "DROP"
    elif "jeopardy" in name:
        return "JPRDY"
    elif "naturalqs" in name:
        return "NatQs"
    elif "squad_v2_conversational" in name:
        return "squad_v2_conv"
    elif "ftrace" in name:
        return "ftrace_conv"
    elif "squad" in name:
        return "SQuAD"
    elif "olmo3:base_easy:math_bpb" == name:
        return "math_bpb"
    elif "olmo3:base_easy:code_bpb" == name:
        return "code_bpb"
    else:
        return name+" TODO"
df["task"] = df["alias"].apply(rename)
df = df[~df['task'].str.contains("TODO", na=False)]
cols = df.columns[:2].tolist() + sorted(df.columns[2:])
df = df[cols]
# df = df[~df["model"].str.contains("lr1e-05")]
df

[PosixPath('finetuning_eval/olmo3:base_easy:math_bpb/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/qa:ftrace_conversational/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42/metrics.json'), PosixPath('finetuning_eval/qa:squad_v2_conversational/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42/metrics.json'), PosixPath('finetuning_eval/olmo3:base_easy:code_bpb/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/olmo_2_generative::olmes/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/qa:hardcoded/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42/metrics.json')]


/tmp/ipykernel_1309157/1510196429.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(all_dfs, ignore_index=True)


,alias,primary_score_x,HasAns_exact,HasAns_f1,NoAns_exact,NoAns_f1,acc_per_byte,acc_per_byte_macro,acc_per_byte_micro,acc_per_char,...,num_instances,primary_score_macro,primary_score_micro,primary_score_y,recall,sum_logits_corr,sum_logits_corr_macro,sum_logits_corr_micro,task,task_suite
0,olmo3:base_easy:math_bpb,0.518897,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,...,5000,0.518897,0.522348,0.518897,NaN,NaN,-191.421902,-182.795005,math_bpb,olmo3:base_easy:math_bpb
8,OLMo-2-0425-1B_ftrace_date-lm_conversational_l...,46.121859,42.024448,46.121859,NaN,NaN,NaN,NaN,NaN,NaN,...,6708,NaN,NaN,46.121859,NaN,NaN,NaN,NaN,ftrace_conv,qa:ftrace_conversational
9,OLMo-2-0425-1B_ftrace_date-lm_conversational_l...,10.362795,9.750337,15.104161,5.634987,5.634987,NaN,NaN,NaN,NaN,...,11873,NaN,NaN,10.362795,NaN,NaN,NaN,NaN,squad_v2_conv,qa:squad_v2_conversational
11,olmo3:base_easy:code_bpb,0.580422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9164,0.580422,0.491221,0.580422,NaN,NaN,NaN,NaN,code_bpb,olmo3:base_easy:code_bpb
31,coqa::olmes,0.611560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7983,NaN,NaN,0.611560,0.539521,NaN,NaN,NaN,CoQA,olmo_2_generative::olmes
32,squad::olmes,0.762038,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1000,NaN,NaN,0.762038,0.689000,NaN,NaN,NaN,SQuAD,olmo_2_generative::olmes
33,jeopardy::olmes,0.598233,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2116,NaN,NaN,0.598233,0.481096,NaN,NaN,NaN,JPRDY,olmo_2_generative::olmes
34,naturalqs::olmes,0.174660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1000,NaN,NaN,0.174660,NaN,NaN,NaN,NaN,NatQs,olmo_2_generative::olmes
35,drop::olmes,0.341250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1000,NaN,NaN,0.341250,NaN,NaN,NaN,NaN,DROP,olmo_2_generative::olmes
36,OLMo-2-0425-1B_ftrace_date-lm_conversational_l...,3.696152,0.000000,3.696152,NaN,NaN,NaN,NaN,NaN,NaN,...,240,NaN,NaN,3.696152,NaN,NaN,NaN,NaN,ftrace_conv,qa:hardcoded


In [27]:
import os
import pandas as pd


df_f = df[["task_suite", "alias", "primary_score_y", "model", "task"]].copy()
df_f = (
    df_f.groupby(["model", "task"])["primary_score_y"]
    .mean()
    .reset_index()
    .pivot(index="model", columns="task", values="primary_score_y")
)

# df_f.insert(0, "Avg", df_f.mean(axis=1, numeric_only=True))

external_models = [
    "Qwen2.5-0.5B-Instruct",
    "OLMo-2-0425-1B-SFT",
    "Llama-3.2-1B-Instruct",
]

df_f = df_f.reset_index()
df_f["type"] = df_f["model"].apply(
    lambda x: "external" if x in external_models else "own"
)


score_cols = [c for c in df_f.columns if c not in ["model", "type"]]
bpb_cols = [c for c in score_cols if "bpb" in c.lower()]
print(bpb_cols)
for col in score_cols:
    if col in bpb_cols:
        min_val = df_f[col].min()
        df_f[col] = df_f[col].apply(lambda x: f"\\textbf{{{x:.2f}}}" if x == min_val else f"{x:.2f}")
    else:
        max_val = df_f[col].max()
        df_f[col] = df_f[col].apply(lambda x: f"\\textbf{{{x:.2f}}}" if x == max_val else f"{x:.2f}")

df_f.rename(
    columns={c: f"\\rotatebox{{45}}{{{c}}}" for c in score_cols},
    inplace=True,
)

df_f["model"] = df_f["model"]#.apply(lambda x: x.split("_")[0])


own = df_f[df_f["type"] == "own"].drop(columns="type")
ext = df_f[df_f["type"] == "external"].drop(columns="type")

final_df = pd.concat([own, ext], axis=0)
final_df = final_df.set_index("model")
final_df.index.name = ""
final_df.columns.name = "Task"
latex = final_df.to_latex(
    escape=False,
    index=True,
    header=True,
    column_format = "l|c|" + "c" * (len(final_df.columns) - 1)
)


n_own = len(own)
n_cols = len(final_df.columns) + 1  # +1 for index column

lines = latex.splitlines()

# first midrule after header
data_start = next(
    i for i, l in enumerate(lines) if l.strip() == "\\midrule"
) + 1

insert_at = data_start + n_own
empty_cells = " & " * (n_cols - 1)

lines.insert(
    data_start,
    f"\\textbf{{Fine-tuned Models}}{empty_cells} \\\\"
)

lines.insert(
    insert_at+1,
    "\\hline\n"
    f"\\textbf{{External Models}}{empty_cells} \\\\"
)



latex = "\n".join(lines)


os.makedirs("tables", exist_ok=True)
with open("tables/olmes-all.tex", "w") as f:
    f.write(latex)


['code_bpb', 'math_bpb']


In [28]:
# !python3 -m wandb sweep --project=eemc_finetuning sweep_code.yaml

In [29]:
final_df

Task,\rotatebox{45}{CoQA},\rotatebox{45}{DROP},\rotatebox{45}{JPRDY},\rotatebox{45}{NatQs},\rotatebox{45}{SQuAD},\rotatebox{45}{code_bpb},\rotatebox{45}{ftrace_conv},\rotatebox{45}{math_bpb},\rotatebox{45}{squad_v2_conv}
,,,,,,,,,
OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42,nan,nan,nan,nan,nan,nan,\textbf{24.91},nan,\textbf{10.36}
OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42-merged,\textbf{0.61},\textbf{0.34},\textbf{0.60},\textbf{0.17},\textbf{0.76},\textbf{0.58},nan,\textbf{0.52},nan


In [30]:
asdf

NameError: name 'asdf' is not defined

In [ ]:
import os
import re
import yaml


def clean_task_name(task_col):
    m = re.match(r"\\?rotatebox\{45\}\{(.+?)\}", task_col)
    if m:
        return m.group(1).replace("\\_", "_")
    return task_col.replace("\\_", "_")

def clean_score(score):
    if pd.isna(score) or score == "":
        return None
    score_str = str(score).replace("\\textbf{", "").replace("}", "").strip()
    try:
        return float(score_str)
    except ValueError:
        return None


task_map = {
    "ARC_E": ("multiple-choice", "ARC-Easy", "arc"),
    "ARC_C": ("multiple-choice", "ARC-Challenge", "arc"),
    "BoolQ": ("question-answering", "BoolQ", "boolq"),
    "CSQA": ("multiple-choice", "CSQA", "commonsense_qa"),
    "HSwag": ("multiple-choice", "HellaSwag", "hellaswag"),
    "MMLU-Pro": ("multiple-choice", "MMLU-Pro", "mmlu"),
    "MMLU": ("multiple-choice", "MMLU", "mmlu"),
    "OBQA": ("multiple-choice", "OpenBookQA", "openbookqa"),
    "PIQA": ("multiple-choice", "PIQA", "piqa"),
    "SIQA": ("multiple-choice", "SocialIQA", "socialiqa"),
    "WinoG": ("coreference-resolution", "Winograd", "winogrande"),
    "AGIEval": ("multiple-choice", "AGIEval", "agi"),
    "BBH": ("multiple-choice", "BBH", "bbh"),
    "GSM8K": ("math-word-problem", "GSM8K", "gsm8k"),
    "TriviaQA": ("question-answering", "TriviaQA", "triviaqa"),
    "CoQA": ("question-answering", "CoQA", "coqa"),
    "DROP": ("question-answering", "DROP", "drop"),
    "JPRDY": ("question-answering", "Jeopardy", "jeopardy"),
    "NatQs": ("question-answering", "Natural Questions", "naturalqs"),
    "SQuAD": ("question-answering", "SQuAD", "squad")
}


MODELS_DIR = "./models"
all_folders = [m for m in os.listdir(MODELS_DIR) if "merged" not in m]




model_index = []

for model_folder in all_folders:
    model_name = model_folder.split("_")[0]
    results = []
    for task_col in final_df.columns:
        if "Avg" in task_col:
            continue
        task_clean = clean_task_name(task_col)
        score_val = clean_score(final_df.loc[model_name, task_col])
        if score_val is None or task_clean not in task_map:
            continue

        task_type, dataset_name, dataset_type = task_map[task_clean]
        results.append({
            "task": {"type": task_type, "name": task_clean},
            "dataset": {"type": dataset_type, "name": dataset_name},
            "metrics": [{"type": "accuracy", "value": score_val}]
        })

    model_entry = {"name": model_name, "results": results}
    model_index.append(model_entry)


    

    os.makedirs(model_folder, exist_ok=True)
    readme_path = os.path.join("./models", model_folder, "README.md")

  
    if os.path.exists(readme_path):
        with open(readme_path, "r", encoding="utf-8") as f:
            readme_content = f.read()
    else:
        readme_content = ""

   
    if f"model-index" not in readme_content:
        


        yaml_str = yaml.dump({"model-index": [model_entry]}, sort_keys=False)
        import yaml
        import re

   
        yaml_str = yaml.dump({"model-index": [model_entry]}, sort_keys=False)

        with open(readme_path, "r", encoding="utf-8") as f:
            content = f.read()

        def insert_between_markers(match):
            return f"---\n{yaml_str}{match.group(1)}\n---"

        new_content = re.sub(
            r"(?s)---\n(.*?)\n---",  
            insert_between_markers,
            content,
            count=1  
        )


        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(new_content)
    if f"## Evaluation" not in readme_content:
        
        def generate_evaluation_table(model_entry):
            table_lines = [
                "## Evaluation",
                "We evaluate with [OLMES](https://github.com/allenai/olmes)",
                "",
                "Task suites: `core_9mcqa::olmes`, `mmlu:mc::olmes`, `olmo_2_generative::olmes`, `olmo_2_heldout::olmes`"
                "",
                "| Task | Score|",
                "|------|---------|"
            ]
            for result in model_entry["results"]:
                task_name = result["task"]["name"]
            
                accuracy = result["metrics"][0]["value"]
                dataset_type = result["dataset"]["type"]

                table_lines.append(f"| {task_name} | {accuracy:.2f} |")

            return "\n".join(table_lines)

        print(generate_evaluation_table(model_entry))
        with open(readme_path, "a", encoding="utf-8") as f:
            f.write("\n\n" + generate_evaluation_table(model_entry) + "\n")